# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Chạy được trên Colab và Kaggle. Ô này clone fork của bạn (chạy lại thì chỉ git pull để lấy code mới).
# Colab: gắn Google Drive và trỏ runs/, predictions/, curves/, eval_out/ sang Drive để Colab ngắt không mất kết quả.
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
os.chdir(WORK_DIR)

REPO_URL = "https://github.com/vuthien3002-sys/K4-Day2-VuDucThien-2A202602437.git"
REPO_DIR = f"{WORK_DIR}/K4-Day2-VuDucThien-2A202602437"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/K4-Day2-2A202602437"
    for d in ["runs", "predictions", "curves", "eval_out"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
        if not os.path.lexists(d):
            os.symlink(f"{PERSIST_DIR}/{d}", d)
else:
    PERSIST_DIR = WORK_DIR   # Kaggle: /kaggle/working được lưu khi Save Version
print("WORK_DIR =", WORK_DIR, "| kết quả lưu ở", PERSIST_DIR)

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# REPO_DIR đặt ở ô trên. CODE_DIR là thư mục bài làm của bạn (chép từ starter/).
CODE_DIR = f"{REPO_DIR}/submissions/2A202602437_vu_duc_thien/code"
sys.path.insert(0, REPO_DIR)    # để import eval.py
sys.path.insert(0, CODE_DIR)    # dataset.py, model.py, train.py... của bạn

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Nhãn và các file chia fold tải từ GitHub của tác giả. Ảnh: `images.zip` từ Zenodo (~490 MB), kiểm MD5 rồi giải nén.
Trên Colab, zip được cache vào Drive sau lần tải đầu, các phiên sau chỉ chép lại từ Drive.
Trên Kaggle, nếu đã **Add Input** dataset chứa 17.509 ảnh thì dùng luôn (không có zip để kiểm MD5, thay bằng kiểm tra tên file ở ô sau).

In [ ]:
import glob, hashlib

DATA_DIR = f"{WORK_DIR}/data" if IN_COLAB else "/tmp/deepweeds"   # Kaggle: ngoài /kaggle/working
os.makedirs(f"{DATA_DIR}/labels", exist_ok=True)

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O {DATA_DIR}/labels/{name}.csv {BASE}/{name}.csv
    assert os.path.getsize(f"{DATA_DIR}/labels/{name}.csv") > 0, "tải nhãn thất bại: kiểm tra Internet"
!ls -la {DATA_DIR}/labels

input_jpgs = glob.glob("/kaggle/input/**/*.jpg", recursive=True) if IN_KAGGLE else []
if input_jpgs:
    IMAGES_DIR = os.path.dirname(input_jpgs[0])
    print("Dùng ảnh từ Kaggle Input:", IMAGES_DIR, "(đã giải nén sẵn, không có zip để kiểm MD5)")
else:
    ZIP = f"{DATA_DIR}/images.zip"
    ZIP_CACHE = f"{PERSIST_DIR}/images.zip" if IN_COLAB else None
    if not os.path.exists(ZIP):
        if ZIP_CACHE and os.path.exists(ZIP_CACHE):
            !cp {ZIP_CACHE} {ZIP}
        else:
            !wget -q -O {ZIP} "https://zenodo.org/records/7939060/files/images.zip?download=1"
    h = hashlib.md5()
    with open(ZIP, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()} (xoá {ZIP} rồi chạy lại ô)"
    print("MD5 OK")
    if ZIP_CACHE and not os.path.exists(ZIP_CACHE):
        !cp {ZIP} {ZIP_CACHE}
    !unzip -q -n {ZIP} -d {DATA_DIR}/images
    IMAGES_DIR = os.path.dirname(glob.glob(f"{DATA_DIR}/images/**/*.jpg", recursive=True)[0])
!ls {IMAGES_DIR} | head

In [ ]:
# Kiểm tra ảnh khớp nhãn: đủ 17.509 dòng và mọi Filename trong labels.csv đều có trong IMAGES_DIR.
# Khi tạo Config ở các bước sau, luôn truyền images_dir=IMAGES_DIR, labels_dir=LABELS_DIR.
LABELS_DIR = f"{DATA_DIR}/labels"
labels_df = pd.read_csv(f"{LABELS_DIR}/labels.csv")
missing = set(labels_df["Filename"]) - set(os.listdir(IMAGES_DIR))
print("IMAGES_DIR =", IMAGES_DIR, "|", len(labels_df), "dòng labels.csv | thiếu", len(missing), "ảnh")
assert len(labels_df) == 17509 and not missing, f"thiếu ảnh, ví dụ: {sorted(missing)[:5]}"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# TODO: đọc split bằng dataset.load_split, chạy dataset.check_split(...), in số ảnh mỗi tập và mỗi lớp.
# TODO: vẽ biểu đồ phân bố lớp; xem >= 3 ảnh mỗi lớp; đối chiếu với Table 1 của bài báo
#       (Negatives 9.106; các loài 1.009-1.125 ảnh).

In [ ]:
# TODO: kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3):
#   1) cố định seed
#   2) loss ban đầu của head mới xấp xỉ -ln(1/9) = 2.197
#   3) overfit được một batch nhỏ tới loss gần 0
#   4) vẽ vài ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn để chắc ảnh và nhãn khớp nhau

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id="B01", backbone=..., seed=0))
#       Gợi ý: from train import Config, run
# TODO: ghi vào bảng: tag trọng số, #params, GMAC, macro-F1 val, top-1 val, thời gian train/epoch.
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py